In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [3]:
!pip install tensorflow -q

In [4]:
# import tensorflow
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
# paths
BASE_PATH = "/content/drive/MyDrive/Zero-Day-IoT-Project"

DATA_PATH   = BASE_PATH + "/01_Data"
MODEL_PATH  = BASE_PATH + "/03_Models"
REPORT_PATH = BASE_PATH + "/04_Reports"

os.makedirs(REPORT_PATH, exist_ok=True)

In [6]:
# Load Dataset + Scaler
df = pd.read_parquet(DATA_PATH + "/master_clean.parquet")
scaler = joblib.load(MODEL_PATH + "/scaler.pkl")

print(df.shape)

(152389, 53)


In [7]:
# Feature columns
feature_cols = [c for c in df.columns if c not in ['Attack_label', 'Attack_type']]

In [9]:
# Train Only on Normal Traffic
normal_df = df[df['Attack_label'] == 0].copy()

X_normal = normal_df[feature_cols]

X_normal_scaled = scaler.transform(X_normal)

print("Normal Data Shape:", X_normal_scaled.shape)

Normal Data Shape: (24152, 51)


In [10]:
# Validation Split
X_train, X_val = train_test_split(
    X_normal_scaled,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_val.shape)

(19321, 51) (4831, 51)


In [11]:
# Build Autoencoder
input_dim = X_train.shape[1]

inputs = Input(shape=(input_dim,))
x = Dense(64, activation='relu')(inputs)
x = Dense(32, activation='relu')(x)
x = Dense(16, activation='relu')(x)
x = Dense(32, activation='relu')(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(input_dim, activation='linear')(x)

autoencoder = Model(inputs, outputs)

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 51)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 51)             │         3,315 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,907 (46.51 KB)

 Trainable params: 11,907 (46.51 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Train Model
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=50,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.5692 - val_loss: 0.3141
Epoch 2/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2683 - val_loss: 0.1715
Epoch 3/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1746 - val_loss: 0.1406
Epoch 4/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1501 - val_loss: 0.1247
Epoch 5/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1331 - val_loss: 0.1156
Epoch 6/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1193 - val_loss: 0.1012
Epoch 7/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1042 - val_loss: 0.0944
Epoch 8/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0873 - val_loss: 0.0784
Epoch 9/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0733 - val_loss: 0.0712
Epoch 10/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0698 - val_loss: 0.0585
Epoch 11/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0632 - val_loss: 0.0445
Epoch 12/50
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0469 - val_lo

In [13]:
# Validation Reconstruction Error
X_val_pred = autoencoder.predict(X_val)

val_loss = np.mean(np.square(X_val - X_val_pred), axis=1)

print("Mean Reconstruction Error:", val_loss.mean())

151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Mean Reconstruction Error: 0.013894589256367803


In [14]:
# Set Threshold
threshold = np.percentile(val_loss, 95)

print("Threshold:", threshold)

Threshold: 0.01217610392741139


In [15]:
# Save Threshold
joblib.dump(threshold, MODEL_PATH + "/ae_threshold.pkl")

print("Threshold Saved")

Threshold Saved


In [22]:
# Save Autoencoder
autoencoder.save(MODEL_PATH + "/autoencoder.keras")

print("Autoencoder Saved")

Autoencoder Saved


In [23]:
# Test on Full Dataset
X_all = scaler.transform(df[feature_cols])

X_pred = autoencoder.predict(X_all)

recon_error = np.mean(np.square(X_all - X_pred), axis=1)

df['AE_Predicted_Label'] = (recon_error > threshold).astype(int)

4763/4763 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step


In [24]:
# Evaluate
print(classification_report(
    df['Attack_label'],
    df['AE_Predicted_Label']
))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97     24152
           1       0.99      1.00      1.00    128237

    accuracy                           0.99    152389
   macro avg       1.00      0.97      0.98    152389
weighted avg       0.99      0.99      0.99    152389



In [25]:
# Confusion Matix
cm = confusion_matrix(
    df['Attack_label'],
    df['AE_Predicted_Label']
)

print(cm)

[[ 22913   1239]
 [     0 128237]]


In [26]:
# Save Matrix
report = classification_report(
    df['Attack_label'],
    df['AE_Predicted_Label'],
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    REPORT_PATH + "/autoencoder_metrics.csv"
)

print("Metrics Saved")

Metrics Saved


In [27]:
print("FILES CREATED")
print("----------------")
print("03_Models/autoencoder.h5")
print("03_Models/ae_threshold.pkl")
print("04_Reports/autoencoder_metrics.csv")

FILES CREATED
----------------
03_Models/autoencoder.h5
03_Models/ae_threshold.pkl
04_Reports/autoencoder_metrics.csv
